# Gaussian HMM Experiment

Notebook for the Gaussian HMM pipeline. Core logic lives in `src/gaussian_hmm`; this notebook only runs the pipeline and displays artifacts.

In [22]:
# import os 
# import sys

# current_file_path = os.getcwd() # parent folder path of the current file
# print(current_file_path)
# print(os.path.dirname(current_file_path))
# sys.path.append(os.path.dirname(current_file_path)) # Add root path to sys.path

In [23]:
import os
from pathlib import Path
# print(Path.cwd())
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(Path.cwd())

d:\Master\2025T2\TTT-stochastic-methods-and-applications


In [24]:
from src.gaussian_hmm.config import load_hmm_config
from src.gaussian_hmm.train import run_training
from src.gaussian_hmm.inference import run_inference
from src.gaussian_hmm.evaluate import run_evaluation
from src.gaussian_hmm.visualize import run_visualization

config = load_hmm_config('configs/hmm.yaml')
config

HMMConfig(config_path=WindowsPath('D:/Master/2025T2/TTT-stochastic-methods-and-applications/configs/hmm.yaml'), project_root=WindowsPath('D:/Master/2025T2/TTT-stochastic-methods-and-applications'), data_path=WindowsPath('D:/Master/2025T2/TTT-stochastic-methods-and-applications/data/processed/market_features.csv'), metadata_path=WindowsPath('D:/Master/2025T2/TTT-stochastic-methods-and-applications/data/processed/market_features_metadata.json'), feature_columns=['log_return_z', 'simple_return_z', 'intraday_return_z', 'intraday_range_z', 'volume_log_change_z', 'volatility_5_z', 'volatility_20_z', 'return_mean_5_z', 'volume_zscore_20_z', 'RSI_z', 'MACD_z', 'Sentiment_z'], date_column='Date', split_column='split', fold_column='walk_forward_fold', price_column='Close', return_column='log_return', volatility_column='volatility_20', volume_column='Volume', train_splits=['train'], validation_splits=['validation'], final_fit_splits=['train', 'validation'], n_components=[2, 3, 4, 5], covariance_t

In [25]:
# Run training, inference, evaluation, and visualization for the Gaussian HMM model using the specified configuration. 
# The training process fits a grid of models with different hyperparameters, selects the best model based on evaluation metrics, and saves the results. 
# Inference generates Viterbi state sequences and posterior probabilities, which are then evaluated and visualized.
model_selection = run_training(config)
model_selection.sort_values(['bic', 'validation_log_likelihood'], ascending=[True, False]).head()

Model is not converging.  Current: 26065.418522845754 is not greater than 26065.423140681705. Delta is -0.004617835951648885
Model is not converging.  Current: 25243.8791866538 is not greater than 25243.958188668927. Delta is -0.07900201512529748


Wrote model selection table to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\tables\gaussian_hmm\model_selection.csv
Wrote best Gaussian HMM artifact to D:\Master\2025T2\TTT-stochastic-methods-and-applications\models\gaussian_hmm\best_hmm.pkl


,n_components,covariance_type,seed,status,train_log_likelihood,train_avg_log_likelihood,validation_log_likelihood,validation_avg_log_likelihood,n_parameters,aic,bic,converged,iterations,min_state_share,max_state_share,state_counts
35,5,full,0,ok,26736.571322,3.380525,-1327.801684,-0.783364,474,-52525.142643,-49218.634000,True,87,0.000885,0.524339,"[4147, 590, 2181, 7, 984]"
36,5,full,1,ok,26736.571290,3.380525,-1327.800944,-0.783363,474,-52525.142581,-49218.633938,True,85,0.000885,0.524339,"[4147, 2181, 7, 984, 590]"
38,5,full,3,ok,26736.571267,3.380525,-1327.800396,-0.783363,474,-52525.142535,-49218.633892,True,84,0.000885,0.524339,"[2181, 4147, 984, 590, 7]"
25,4,full,0,ok,25708.772385,3.250572,-1504.142981,-0.887400,375,-50667.544769,-48051.636033,True,78,0.060690,0.529270,"[2158, 1085, 480, 4186]"
28,4,full,3,ok,25708.772382,3.250572,-1504.143018,-0.887400,375,-50667.544764,-48051.636028,True,72,0.060690,0.529270,"[480, 1085, 4186, 2158]"


In [26]:
# Run inference with the best Gaussian HMM model, generating Viterbi state sequences and posterior probabilities. The function loads the trained model artifact, processes the market data to create a feature matrix, and uses the model to predict states and compute posterior probabilities. It then builds inference tables containing the results and saves them to CSV files in the specified output directory.
state_sequence, posterior = run_inference(config)
state_sequence.head()

Wrote Viterbi state sequence to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\tables\gaussian_hmm\state_sequence.csv
Wrote posterior probabilities to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\tables\gaussian_hmm\posterior_probabilities.csv


,Date,split,walk_forward_fold,Close,log_return,volatility_20,Volume,state,state_probability
0,2010-01-29,train,NaN,95.681524,0.018662,0.021043,1338964,2,1.000000
1,2010-02-01,train,NaN,95.660761,-0.000217,0.020988,4358805,2,1.000000
2,2010-02-02,train,NaN,95.136289,-0.005498,0.019919,4710519,2,0.999998
3,2010-02-03,train,NaN,91.935683,-0.034221,0.019597,2527682,2,1.000000
4,2010-02-04,train,NaN,91.676653,-0.002821,0.019599,1838577,2,0.999999


In [27]:
# Run evaluation of the Gaussian HMM model's performance using the predicted state sequences and posterior probabilities. The function computes various evaluation metrics, such as accuracy, precision, recall, and F1-score, and returns a summary of the model's performance.
evaluation = run_evaluation(config)
evaluation['state_summary']

Wrote transition matrix to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\tables\gaussian_hmm\transition_matrix.csv
Wrote state summary to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\tables\gaussian_hmm\state_summary.csv
Wrote walk-forward results to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\tables\gaussian_hmm\walk_forward_results.csv


,state,observations,share,mean_log_return,median_log_return,std_log_return,mean_feature_volatility,mean_volume,mean_assigned_probability,financial_profile
0,0,4410,0.390300,0.000102,0.000179,0.009827,0.009852,2.543046e+06,0.994642,sideways_or_mixed
1,1,2104,0.186211,-0.001623,-0.000648,0.076775,0.087751,2.585822e+06,0.984056,sideways_or_mixed
2,2,3962,0.350651,-0.000500,-0.000186,0.026155,0.027510,2.576959e+06,0.986718,sideways_or_mixed
3,3,823,0.072838,0.004996,0.020796,0.448698,0.399908,2.583498e+06,0.991634,volatile


In [28]:
run_visualization(config)

Wrote figure to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\figures\gaussian_hmm\price_states.png
Wrote figure to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\figures\gaussian_hmm\posterior_probabilities.png
Wrote figure to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\figures\gaussian_hmm\transition_matrix.png
Wrote figure to D:\Master\2025T2\TTT-stochastic-methods-and-applications\reports\figures\gaussian_hmm\aic_bic.png


['D:\\Master\\2025T2\\TTT-stochastic-methods-and-applications\\reports\\figures\\gaussian_hmm\\price_states.png',
 'D:\\Master\\2025T2\\TTT-stochastic-methods-and-applications\\reports\\figures\\gaussian_hmm\\posterior_probabilities.png',
 'D:\\Master\\2025T2\\TTT-stochastic-methods-and-applications\\reports\\figures\\gaussian_hmm\\transition_matrix.png',
 'D:\\Master\\2025T2\\TTT-stochastic-methods-and-applications\\reports\\figures\\gaussian_hmm\\aic_bic.png']